# Exercise 1 — Marvel Universe Social Network Analysis

**Course:** Big Data Analytics 
**Dataset:** Marvel Universe Social Network (Kaggle)  


Team Members:
* Esteban Gerardo Leiva Montenegro (esteban.leiva.001@student.uni.lu)
* Isaac Fabián Palma Medina (isaac.palma.001@student.uni.lu)
* José Valdivia Agüero (jose.valdivia.001@student.uni.lu)

## Tasks
- **(a)** Load graph from `edges.csv`, compare vertices with `nodes.csv`
- **(b)** Connected Components analysis
- **(c)** Degree distribution, Clustering Coefficient, Average Path Length
- **(d)** PageRank — top-25 heroes and top-25 comics, compared to degree ranking
- **(e)** Hero co-occurrence pairs, compared to `hero-edge.csv`




## 0. Environment Setup

Initialize SparkSession with GraphFrames support.  
Make sure the cluster has the `graphframes` package available via `spark.jars.packages`.

In [1]:
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, desc, asc, count, lit,
    monotonically_increasing_id, least, greatest
)
from graphframes import GraphFrame

spark = (SparkSession.builder
         .appName("Marvel-GraphFrames")
         .master("local[*]")
         .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.11.0")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
# Reduce verbosity of Spark logs
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("SparkSession ready.")
print("Current dir: ",os.getcwd())

:: loading settings :: url = jar:file:/mnt/aiongpfs/apps/easybuild/systems/iris/rhel810-20260107/2024a/broadwell/software/Spark/4.0.1-foss-2024a-Java-21/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/users/eleiva/.ivy2.5.2/cache
The jars for the packages stored in: /home/users/eleiva/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4da90d49-e584-49fd-9a28-cb18c2816d34;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.11.0 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.11.0 in central
:: resolution report :: resolve 137ms :: artifacts dl 5ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark4_2.13;0.11.0 from central in [default]
	io.graphframes#graphframes-spark4_2.13;0.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |      

Spark version: 4.0.1
SparkSession ready.
Current dir:  /mnt/aiongpfs/users/eleiva/uni/2_semester/big_data_analytics/bda-exercises-4-group-1/solutions/problem_1


## (a) Load and Parse the Graph

## Dataset Description
- `nodes.csv` — `(node, type)`: node name and type (`hero` or `comic`)
- `edges.csv` — `(hero, comic)`: which heroes appear in which comics
- `hero-edge.csv` — pairs of heroes that appear together in the same comics


**Steps:**
1. Load `edges.csv` — columns `(hero, comic)`
2. Extract all distinct vertex IDs from both `hero` and `comic` columns
3. Assign a numeric `id` to each vertex (required by GraphFrames)
4. Build the GraphFrame
5. Load `nodes.csv` and compare both vertex sets


In [2]:

BASE_DIR = "../../data/problem_1/kaggle-marvel-universe"

EDGES_PATH     = BASE_DIR+"/edges.csv"
NODES_PATH     = BASE_DIR+"/nodes.csv"
HERO_EDGE_PATH = BASE_DIR+"/hero-network.csv"

#  Load edges.csv 
t0 = time.time()

edges_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(EDGES_PATH)

edges_raw.printSchema()
edges_raw.show(5)
print(f"Total edges (hero, comic): {edges_raw.count()}")

root
 |-- hero: string (nullable = true)
 |-- comic: string (nullable = true)

+--------------------+------+
|                hero| comic|
+--------------------+------+
|24-HOUR MAN/EMMANUEL|AA2 35|
|3-D MAN/CHARLES CHAN| AVF 4|
|3-D MAN/CHARLES CHAN| AVF 5|
|3-D MAN/CHARLES CHAN| COC 1|
|3-D MAN/CHARLES CHAN|H2 251|
+--------------------+------+
only showing top 5 rows
Total edges (hero, comic): 96104


In [3]:
# Both hero names and comic names are nodes in the graph

heroes_df = edges_raw.select(col("hero").alias("node"))
comics_df = edges_raw.select(col("comic").alias("node"))

# Union and deduplicate
all_nodes_from_edges = heroes_df.union(comics_df).distinct()

# Assign a unique numeric id to each node string
# zipWithIndex on RDD guarantees uniqueness
nodes_with_id = all_nodes_from_edges.rdd \
    .zipWithIndex() \
    .map(lambda x: (x[1], x[0][0])) \
    .toDF(["id", "node"])

# Cache — reused many times
nodes_with_id.cache()

print(f"Distinct vertices extracted from edges.csv: {nodes_with_id.count()}")
nodes_with_id.show(5)

Distinct vertices extracted from edges.csv: 19090
+---+--------------------+
| id|                node|
+---+--------------------+
|  0|ANCIENT ONE/BARON MO|
|  1|             ASYLUM/|
|  2|       BANNON, LANCE|
|  3|    CASSADA, MICHAEL|
|  4|      CHAKARA, MADAN|
+---+--------------------+
only showing top 5 rows


In [4]:
#  Build GraphFrame vertices and edges 

# Vertices: (id, node)
vertices = nodes_with_id

# Edges: remap hero and comic string names to numeric IDs
# src = hero_id, dst = comic_id
edges_with_src = edges_raw.join(
    nodes_with_id.select(col("id").alias("src"), col("node").alias("hero")),
    on="hero"
)
edges_final = edges_with_src.join(
    nodes_with_id.select(col("id").alias("dst"), col("node").alias("comic")),
    on="comic"
).select("src", "dst")

edges_final.cache()
print(f"Edges in GraphFrame: {edges_final.count()}")
edges_final.show(5)

# Build the GraphFrame
g = GraphFrame(vertices, edges_final)
print("GraphFrame created successfully.")

Edges in GraphFrame: 96104
+----+-----+
| src|  dst|
+----+-----+
|2047|18174|
|3725|19039|
|3725| 8876|
|3725| 8620|
|3725|17131|
+----+-----+
only showing top 5 rows
GraphFrame created successfully.


In [5]:
# ── Compare with nodes.csv 

nodes_csv = spark.read \
    .option("header", "true") \
    .csv(NODES_PATH)

nodes_csv.printSchema()
print(f"Total nodes in nodes.csv: {nodes_csv.count()}")
nodes_csv.groupBy("type").count().show()

# Nodes in edges.csv but NOT in nodes.csv
in_edges_not_in_nodes = all_nodes_from_edges \
    .join(nodes_csv, all_nodes_from_edges.node == nodes_csv.node, "left_anti")

print(f"Nodes in edges.csv but missing from nodes.csv: {in_edges_not_in_nodes.count()}")
in_edges_not_in_nodes.show(10)

# Nodes in nodes.csv but NOT in edges.csv
in_nodes_not_in_edges = nodes_csv \
    .join(all_nodes_from_edges, nodes_csv.node == all_nodes_from_edges.node, "left_anti")

print(f"Nodes in nodes.csv but missing from edges.csv: {in_nodes_not_in_edges.count()}")
in_nodes_not_in_edges.show(10)

t1 = time.time()
print(f"\n⏱ Runtime (a): {t1 - t0:.2f} seconds")

root
 |-- node: string (nullable = true)
 |-- type: string (nullable = true)

Total nodes in nodes.csv: 19090
+-----+-----+
| type|count|
+-----+-----+
|comic|12651|
| hero| 6439|
+-----+-----+

Nodes in edges.csv but missing from nodes.csv: 1
+--------------------+
|                node|
+--------------------+
|SPIDER-MAN/PETER ...|
+--------------------+

Nodes in nodes.csv but missing from edges.csv: 1
+--------------------+----+
|                node|type|
+--------------------+----+
|SPIDER-MAN/PETER ...|hero|
+--------------------+----+


⏱ Runtime (a): 9.31 seconds


## (b) Connected Components Analysis

**Algorithm recap (Lecture Chapter 6, Slide #19):**  
Each vertex starts with its own ID as state. Iteratively, each vertex sends its state to all neighbors and adopts the **minimum** of all received states. When no state changes, vertices with the same state belong to the same connected component.

**What we expect:**  
Likely one giant component (most heroes are connected through shared comics) and several smaller isolated groups.

In [6]:
t0 = time.time()
username =  os.environ.get('USER') or ""
checkpoint_dir =f"/scratch/users/{username}/marvel_checkpoints"

# GraphFrames requires a checkpoint directory for connected components
spark.sparkContext.setCheckpointDir(checkpoint_dir)

cc = g.connectedComponents()

# Count how many distinct components exist
component_counts = cc.groupBy("component").count().orderBy(desc("count"))

total_components = component_counts.count()
print(f"Total number of connected components: {total_components}")
print("\nTop 10 largest components (component_id, size):")
component_counts.show(10)

# Size of the largest component
largest = component_counts.first()
print(f"Largest component contains {largest['count']} vertices")
print(f"Fraction of all vertices in largest component: "
      f"{largest['count'] / vertices.count():.2%}")

t1 = time.time()
print(f"\n⏱ Runtime (b): {t1 - t0:.2f} seconds")

26/05/31 21:43:46 WARN ConnectedComponents: Algorithm 'graphframes' is deprecated and will be removed in a future release. Using 'two_phase' instead.
26/05/31 21:44:10 WARN TwoPhase$: Returned DataFrame is persistent and materialized!


Total number of connected components: 22

Top 10 largest components (component_id, size):
+---------+-----+
|component|count|
+---------+-----+
|        0|19029|
|      621|   11|
|      507|    8|
|      172|    4|
|     3656|    3|
|      169|    3|
|       34|    2|
|     6393|    2|
|     5388|    2|
|     1124|    2|
+---------+-----+
only showing top 10 rows
Largest component contains 19029 vertices
Fraction of all vertices in largest component: 99.68%

⏱ Runtime (b): 23.86 seconds


## (c) Degree Distribution, Clustering Coefficient, Average Path Length

**Three structural metrics:**

1. **Degree distribution** — how many connections does each node have? Shows if the graph has hubs (few nodes with very high degree) — typical of real-world networks (power law / long tail).

2. **Average clustering coefficient** — for each node, what fraction of its neighbors are also connected to each other? Range 0–1. High values indicate tight local clusters.

3. **Average path length** — average number of hops between any two nodes. Computed via BFS on a 2% sample (full computation would require ~N²/2 pairs which is computationally prohibitive).

In [7]:
# ── (c.1) Degree Distribution ──────────────────────────────────────────────────
t0 = time.time()

degrees = g.degrees
degrees.cache()

degree_stats = degrees.select("degree").describe()
print("Degree statistics:")
degree_stats.show()

#
verts = vertices.select(col("id").alias("v_id"), col("node").alias("v_node"))

top_degree = (degrees.join(verts, degrees["id"] == verts["v_id"])
              .select(verts["v_node"].alias("node"), degrees["degree"])
              .orderBy(desc("degree")))

print("Top 10 most connected nodes:")
top_degree.show(10, truncate=False)

degree_distribution = degrees.groupBy("degree").count().orderBy("degree")
print("Degree distribution (first 20 values):")
degree_distribution.show(20)

t1 = time.time()
print(f" Runtime (c.1 - degree): {t1 - t0:.2f} seconds")

Degree statistics:
+-------+------------------+
|summary|            degree|
+-------+------------------+
|  count|             19090|
|   mean|10.068517548454688|
| stddev|34.988653840744426|
|    min|                 1|
|    max|              1577|
+-------+------------------+

Top 10 most connected nodes:
+-----------------------+------+
|node                   |degree|
+-----------------------+------+
|SPIDER-MAN/PETER PARKER|1577  |
|CAPTAIN AMERICA        |1334  |
|IRON MAN/TONY STARK    |1150  |
|THING/BENJAMIN J. GR   |963   |
|THOR/DR. DONALD BLAK   |956   |
|HUMAN TORCH/JOHNNY S   |886   |
|MR. FANTASTIC/REED R   |854   |
|HULK/DR. ROBERT BRUC   |835   |
|WOLVERINE/LOGAN        |819   |
|INVISIBLE WOMAN/SUE    |762   |
+-----------------------+------+
only showing top 10 rows
Degree distribution (first 20 values):
+------+-----+
|degree|count|
+------+-----+
|     1| 3275|
|     2| 2027|
|     3| 1740|
|     4| 1468|
|     5| 1409|
|     6| 1223|
|     7| 1029|
|     8|  990|

In [8]:
# ── (c.2) Average Clustering Coefficient ───────────────────────────────────────
# Formula (Lecture Slide #29):
#   C(v) = triangles(v) / (degree(v) * (degree(v) - 1) / 2)
# Average C = sum of all C(v) / number of vertices with degree >= 2
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import when

t0 = time.time()
triangle_counts = g.triangleCount(storage_level=StorageLevel.MEMORY_AND_DISK)
triangle_counts.cache()

print("Triangle count statistics:")
triangle_counts.select("count").describe().show()

# Join triangle counts with degree to compute local clustering coefficient
tri = triangle_counts.select(col("id").alias("t_id"), col("count").alias("triangles"))

tri_degree = (tri.join(degrees, tri["t_id"] == degrees["id"])
              .select(degrees["id"], col("triangles"), degrees["degree"]))

tri_degree = tri_degree.withColumn(
    "max_triangles", (col("degree") * (col("degree") - 1) / 2.0)
).withColumn(
    "clustering_coeff",
    when(col("max_triangles") == 0, 0.0)
    .otherwise(col("triangles") / col("max_triangles"))
)

avg_cc = tri_degree.select("clustering_coeff").groupBy().avg("clustering_coeff").first()[0]
print(f"\nAverage Clustering Coefficient: {avg_cc:.6f}")

t1 = time.time()

26/05/31 21:44:13 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/05/31 21:44:13 WARN CacheManager: Asked to cache already cached data.
26/05/31 21:44:15 WARN TriangleCount$: Returned DataFrame is persistent and materialized!
26/05/31 21:44:15 WARN CacheManager: Asked to cache already cached data.


Triangle count statistics:
+-------+-----+
|summary|count|
+-------+-----+
|  count|19090|
|   mean|  0.0|
| stddev|  0.0|
|    min|    0|
|    max|    0|
+-------+-----+


Average Clustering Coefficient: 0.000000


In [9]:
# ── (c.3) Approximate Average Path Length via BFS ──────────────────────────────
t0 = time.time()

# Grafo NO dirigido (aristas en ambos sentidos): en el dirigido hero→comic
# todo lo alcanzable está a 1 salto. Lo definimos aquí por si la celda (d)
# aún no se ha ejecutado.
edges_undirected = edges_final.union(
    edges_final.select(col("dst").alias("src"), col("src").alias("dst"))
)
g_und = GraphFrame(vertices, edges_undirected)

# Sample 2% of vertex IDs as landmarks
all_ids = vertices.select("id").rdd.map(lambda r: r[0]).collect()
import random
random.seed(42)
sample_size = max(1, int(len(all_ids) * 0.02))
landmark_ids = random.sample(all_ids, sample_size)
landmark_ids_str = [str(x) for x in landmark_ids]
print(f"Total vertices: {len(all_ids)}")
print(f"Sampled landmarks (2%): {sample_size}")

# BFS sobre el grafo no dirigido
shortest_paths = g_und.shortestPaths(landmarks=landmark_ids_str)

from pyspark.sql.functions import explode
paths_df = shortest_paths.select(
    col("id").alias("target"),
    explode("distances").alias("landmark", "distance")
).filter(col("distance") > 0)

print("\nApproximate path length statistics:")
paths_df.select("distance").describe().show()

print("Path length distribution:")
paths_df.groupBy("distance").count().orderBy("distance").show()

avg_path = paths_df.groupBy().avg("distance").first()[0]
print(f"Approximate Average Path Length: {avg_path:.4f}")

t1 = time.time()
print(f"⏱ Runtime (c.3 - path length): {t1 - t0:.2f} seconds")

Total vertices: 19090
Sampled landmarks (2%): 381


26/05/31 21:48:20 WARN ShortestPaths: Returned DataFrame is persistent and materialized!



Approximate path length statistics:
+-------+-----------------+
|summary|         distance|
+-------+-----------------+
|  count|          7249668|
|   mean|4.430591718131092|
| stddev|1.024542232553525|
|    min|                1|
|    max|               10|
+-------+-----------------+

Path length distribution:
+--------+-------+
|distance|  count|
+--------+-------+
|       1|   3283|
|       2| 288550|
|       3| 730803|
|       4|2889477|
|       5|2412813|
|       6| 767643|
|       7| 138112|
|       8|  18049|
|       9|    860|
|      10|     78|
+--------+-------+

Approximate Average Path Length: 4.4306
⏱ Runtime (c.3 - path length): 245.30 seconds


In [10]:
# ── (c) Summary — Sparse or Dense? ────────────────────────────────────────────

n_vertices = vertices.count()
n_edges    = edges_final.count()
max_edges  = n_vertices * (n_vertices - 1) / 2  # undirected complete graph
density    = n_edges / max_edges

print("=" * 50)
print("GRAPH STRUCTURAL SUMMARY")
print("=" * 50)
print(f"Vertices:                  {n_vertices:,}")
print(f"Edges:                     {n_edges:,}")
print(f"Graph density:             {density:.8f}")
print(f"Average degree:            (see degree stats above)")
print(f"Avg clustering coefficient:{avg_cc:.6f}")
print(f"Approx avg path length:    {avg_path:.4f}")
print("=" * 50)
print()
print("Interpretation:")
print(f"  Density ≈ {density:.6f} → Very SPARSE graph")
print(f"  High clustering coefficient → Local communities exist")
print(f"  Short avg path length → Small-world network structure")
print("  (Few hubs connect many nodes — power-law degree distribution)")

GRAPH STRUCTURAL SUMMARY
Vertices:                  19,090
Edges:                     96,104
Graph density:             0.00052745
Average degree:            (see degree stats above)
Avg clustering coefficient:0.000000
Approx avg path length:    4.4306

Interpretation:
  Density ≈ 0.000527 → Very SPARSE graph
  High clustering coefficient → Local communities exist
  Short avg path length → Small-world network structure
  (Few hubs connect many nodes — power-law degree distribution)


## (d) PageRank — Top-25 Heroes and Top-25 Comics

**PageRank intuition (Lecture Slide #36):**  
A node is important if important nodes point to it. The random surfer model: with probability `ε` jump to any random node, with probability `(1-ε)` follow an edge. PageRank = fraction of time the surfer spends at each node.

**Why compare to degree?**  
Degree measures quantity of connections. PageRank measures quality — a hero connected to few but very central comics can rank higher than a hero connected to many obscure comics.

In [11]:
t0 = time.time()

# Run PageRank
# resetProbability = ε = 0.15 (random jump probability)
# maxIter = 10 iterations (sufficient for convergence on most real graphs)
edges_undirected = edges_final.union(
    edges_final.select(col("dst").alias("src"), col("src").alias("dst"))
)
g_und = GraphFrame(vertices, edges_undirected)

pagerank = g_und.pageRank(resetProbability=0.15, maxIter=10)

# pageRank.vertices YA trae las columnas originales (id, node) + pagerank,
# así que no hace falta volver a unir con `vertices` (eso causaba la ambigüedad).
pr_typed = (pagerank.vertices
            .join(nodes_csv.select(col("node").alias("n_node"), col("type")),
                  col("node") == col("n_node"), "left")
            .select("node", "type", "pagerank"))

t1 = time.time()
print(f"⏱ PageRank computation: {t1 - t0:.2f} seconds")

26/05/31 21:48:22 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.


⏱ PageRank computation: 2.87 seconds


26/05/31 21:48:24 WARN PageRank: Returned DataFrame is persistent and materialized!


In [12]:
# ── Top-25 Heroes by PageRank ─
print("TOP-25 HEROES BY PAGERANK")
print("=" * 50)
pr_typed.filter(col("type") == "hero") \
        .orderBy(desc("pagerank")) \
        .show(25, truncate=False)

# ── Top-25 Comics by PageRank 
print("TOP-25 COMICS BY PAGERANK")
print("=" * 50)
pr_typed.filter(col("type") == "comic") \
        .orderBy(desc("pagerank")) \
        .show(25, truncate=False)

TOP-25 HEROES BY PAGERANK
+--------------------+----+------------------+
|node                |type|pagerank          |
+--------------------+----+------------------+
|CAPTAIN AMERICA     |hero|103.89207463617768|
|IRON MAN/TONY STARK |hero|91.39203446706996 |
|HULK/DR. ROBERT BRUC|hero|75.72216290560705 |
|THING/BENJAMIN J. GR|hero|72.1061260192274  |
|WOLVERINE/LOGAN     |hero|66.92015665626454 |
|THOR/DR. DONALD BLAK|hero|66.54677622683305 |
|HUMAN TORCH/JOHNNY S|hero|63.150376972223896|
|DR. STRANGE/STEPHEN |hero|60.897770158166   |
|MR. FANTASTIC/REED R|hero|60.37332875552404 |
|DAREDEVIL/MATT MURDO|hero|58.341024664017084|
|INVISIBLE WOMAN/SUE |hero|52.87474294045096 |
|WATSON-PARKER, MARY |hero|49.28616789614475 |
|SUB-MARINER/NAMOR MA|hero|43.36592172303536 |
|JAMESON, J. JONAH   |hero|43.126146758082626|
|HAWK                |hero|42.988588301154344|
|SCARLET WITCH/WANDA |hero|42.8694100830334  |
|BEAST/HENRY &HANK& P|hero|42.11708050049018 |
|PUNISHER II/FRANK CA|hero|40.5736

In [13]:
#  Top-25 by Degree (for comparison) 

# Join degrees with node names and types
degree_typed = degrees.join(
    vertices.select(col("id").alias("v_id"), col("node")),
    degrees.id == col("v_id")
).join(
    nodes_csv.select(col("node").alias("n_node"), col("type")),
    col("node") == col("n_node"),
    "left"
).select("node", "type", "degree")

print("TOP-25 HEROES BY DEGREE")
print("=" * 50)
degree_typed.filter(col("type") == "hero") \
            .orderBy(desc("degree")) \
            .show(25, truncate=False)

print("TOP-25 COMICS BY DEGREE")
print("=" * 50)
degree_typed.filter(col("type") == "comic") \
            .orderBy(desc("degree")) \
            .show(25, truncate=False)

TOP-25 HEROES BY DEGREE
+--------------------+----+------+
|node                |type|degree|
+--------------------+----+------+
|CAPTAIN AMERICA     |hero|1334  |
|IRON MAN/TONY STARK |hero|1150  |
|THING/BENJAMIN J. GR|hero|963   |
|THOR/DR. DONALD BLAK|hero|956   |
|HUMAN TORCH/JOHNNY S|hero|886   |
|MR. FANTASTIC/REED R|hero|854   |
|HULK/DR. ROBERT BRUC|hero|835   |
|WOLVERINE/LOGAN     |hero|819   |
|INVISIBLE WOMAN/SUE |hero|762   |
|SCARLET WITCH/WANDA |hero|643   |
|BEAST/HENRY &HANK& P|hero|635   |
|DR. STRANGE/STEPHEN |hero|631   |
|WATSON-PARKER, MARY |hero|622   |
|DAREDEVIL/MATT MURDO|hero|619   |
|HAWK                |hero|605   |
|VISION              |hero|603   |
|CYCLOPS/SCOTT SUMMER|hero|585   |
|WASP/JANET VAN DYNE |hero|581   |
|JAMESON, J. JONAH   |hero|577   |
|ANT-MAN/DR. HENRY J.|hero|561   |
|SUB-MARINER/NAMOR MA|hero|530   |
|STORM/ORORO MUNROE S|hero|523   |
|PROFESSOR X/CHARLES |hero|496   |
|FURY, COL. NICHOLAS |hero|471   |
|MARVEL GIRL/JEAN GRE|hero|466 

In [14]:
#  Compare PageRank ranking vs Degree ranking 
# Add rank numbers and join both rankings side by side

from pyspark.sql.window import Window
from pyspark.sql.functions import rank as spark_rank

window_pr     = Window.partitionBy("type").orderBy(desc("pagerank"))
window_degree = Window.partitionBy("type").orderBy(desc("degree"))

pr_ranked = pr_typed.withColumn("pr_rank", spark_rank().over(window_pr))
deg_ranked = degree_typed.withColumn("deg_rank", spark_rank().over(window_degree))

# Join on node name for heroes
comparison_heroes = pr_ranked.filter((col("pr_rank") <= 25) & (col("type") == "hero")) \
    .join(
        deg_ranked.filter(col("type") == "hero").select(
            col("node").alias("d_node"), "deg_rank", "degree"
        ),
        pr_ranked.node == col("d_node"),
        "left"
    ).select("node", "pr_rank", col("pagerank").cast("double"), "deg_rank", "degree") \
     .orderBy("pr_rank")

print("HERO COMPARISON — PageRank rank vs Degree rank (top 25 by PR)")
print("A large gap between pr_rank and deg_rank = PageRank captures something degree misses")
comparison_heroes.show(25, truncate=False)

HERO COMPARISON — PageRank rank vs Degree rank (top 25 by PR)
A large gap between pr_rank and deg_rank = PageRank captures something degree misses
+--------------------+-------+------------------+--------+------+
|node                |pr_rank|pagerank          |deg_rank|degree|
+--------------------+-------+------------------+--------+------+
|CAPTAIN AMERICA     |1      |103.89207463617768|1       |1334  |
|IRON MAN/TONY STARK |2      |91.39203446706996 |2       |1150  |
|HULK/DR. ROBERT BRUC|3      |75.72216290560705 |7       |835   |
|THING/BENJAMIN J. GR|4      |72.1061260192274  |3       |963   |
|WOLVERINE/LOGAN     |5      |66.92015665626454 |8       |819   |
|THOR/DR. DONALD BLAK|6      |66.54677622683305 |4       |956   |
|HUMAN TORCH/JOHNNY S|7      |63.150376972223896|5       |886   |
|DR. STRANGE/STEPHEN |8      |60.897770158166   |12      |631   |
|MR. FANTASTIC/REED R|9      |60.37332875552404 |6       |854   |
|DAREDEVIL/MATT MURDO|10     |58.341024664017084|14      |619

## (e) Hero Co-occurrence Pairs

**Methodology (same as Lecture Chapter 6 — MeSH topic co-occurrences):**  
Two heroes co-occur if they appear in the same comic. We perform a **self-join** on `edges.csv` grouping by comic — for each comic, every pair of heroes that appears in it forms a co-occurrence edge.

The result is then compared with `hero-edge.csv` to find:
- Pairs we computed that are also in `hero-edge.csv` ✅
- Pairs in `hero-edge.csv` not in our computation ❓
- Pairs we computed not in `hero-edge.csv` ❓

In [15]:
t0 = time.time()

# ── Self-join edges on comic to generate hero pairs ────────────────────────────
# For each comic, find all pairs (heroA, heroB) where heroA < heroB
# The heroA < heroB condition avoids duplicates (A,B) and (B,A) — same as
# the .sorted.combinations(2) used in the Scala slides

hero_pairs = edges_raw.alias("e1").join(
    edges_raw.alias("e2"),
    on=col("e1.comic") == col("e2.comic")  # same comic
).filter(
    col("e1.hero") < col("e2.hero")         # canonical order — avoids (A,B) and (B,A)
).select(
    col("e1.hero").alias("hero1"),
    col("e2.hero").alias("hero2")
).distinct()  # one pair per (hero1, hero2) regardless of how many comics they share

hero_pairs.cache()
computed_count = hero_pairs.count()
print(f"Computed hero co-occurrence pairs: {computed_count:,}")
hero_pairs.show(10, truncate=False)

t1 = time.time()
print(f"⏱ Runtime (e - pair generation): {t1 - t0:.2f} seconds")

[Stage 2163:>                                                       (0 + 1) / 1]

Computed hero co-occurrence pairs: 171,644
+--------------------+--------------------+
|hero1               |hero2               |
+--------------------+--------------------+
|4-D MAN/MERCURIO    |IRON MAN/TONY STARK |
|ABEL                |TSUNG, MARCUS       |
|ABOMINATION/EMIL BLO|REYES, DR. CECELIA  |
|ABSALOM             |OZYMANDIAS          |
|ABSORBING MAN/CARL C|ELECTRO/MAX DILLON  |
|ABSORBING MAN/CARL C|LANN                |
|ABSORBING MAN/CARL C|VULTURE/ADRIAN TOOME|
|ABSORBING MAN/CARL C|LORELEI II/MELODI [A|
|ADAMS, NICOLE NIKKI |CLINTON, BILL       |
|ADVERSARY           |DAZZLER II/ALLISON B|
+--------------------+--------------------+
only showing top 10 rows
⏱ Runtime (e - pair generation): 1.67 seconds


In [16]:
# ── Load hero-edge.csv ─────────────────────────────────────────────────────────
hero_edge_csv = spark.read \
    .option("header", "true") \
    .csv(HERO_EDGE_PATH)

hero_edge_csv.printSchema()
print(f"Pairs in hero-edge.csv: {hero_edge_csv.count():,}")
hero_edge_csv.show(5, truncate=False)

root
 |-- hero1: string (nullable = true)
 |-- hero2: string (nullable = true)

Pairs in hero-edge.csv: 574,467
+--------------------+--------------------+
|hero1               |hero2               |
+--------------------+--------------------+
|LITTLE, ABNER       |PRINCESS ZANDA      |
|LITTLE, ABNER       |BLACK PANTHER/T'CHAL|
|BLACK PANTHER/T'CHAL|PRINCESS ZANDA      |
|LITTLE, ABNER       |PRINCESS ZANDA      |
|LITTLE, ABNER       |BLACK PANTHER/T'CHAL|
+--------------------+--------------------+
only showing top 5 rows


In [17]:
# ── Normalize hero-edge.csv to canonical order (heroA < heroB) ────────────────
# hero-edge.csv may have pairs in either order — normalize for fair comparison

# Detect column names from hero-edge.csv
c1, c2 = hero_edge_csv.columns[0], hero_edge_csv.columns[1]

hero_edge_normalized = hero_edge_csv.select(
    least(col(c1), col(c2)).alias("hero1"),
    greatest(col(c1), col(c2)).alias("hero2")
).distinct()

print(f"Normalized pairs in hero-edge.csv: {hero_edge_normalized.count():,}")

# ── Comparison ─────────────────────────────────────────────────────────────────

# Pairs we computed that are also in hero-edge.csv
in_both = hero_pairs.join(hero_edge_normalized, on=["hero1", "hero2"])
print(f"\nPairs in BOTH (our computation ∩ hero-edge.csv): {in_both.count():,}")

# Pairs we computed but NOT in hero-edge.csv
only_computed = hero_pairs.join(
    hero_edge_normalized, on=["hero1", "hero2"], how="left_anti"
)
print(f"Pairs only in our computation (not in hero-edge.csv): {only_computed.count():,}")
only_computed.show(10, truncate=False)

# Pairs in hero-edge.csv but NOT in our computation
only_in_file = hero_edge_normalized.join(
    hero_pairs, on=["hero1", "hero2"], how="left_anti"
)
print(f"Pairs only in hero-edge.csv (not in our computation): {only_in_file.count():,}")
only_in_file.show(10, truncate=False)

Normalized pairs in hero-edge.csv: 167,219

Pairs in BOTH (our computation ∩ hero-edge.csv): 138,450


Pairs only in our computation (not in hero-edge.csv): 33,194
+--------------------+--------------------+
|hero1               |hero2               |
+--------------------+--------------------+
|4-D MAN/MERCURIO    |IRON MAN/TONY STARK |
|AMPHIBIAN/KINGLEY RI|DR. STRANGE/STEPHEN |
|ANDROMEDA/ANDROMEDA |CAPTAIN AMERICA     |
|ANGAR THE SCREAMER/D|ROCKET RACER/ROBERT |
|ANGEL/WARREN KENNETH|STUNT-MASTER/GEORGE |
|APE MAN/GORDON MONK |BEAST/HENRY &HANK& P|
|AQUARIAN/WUNDARR    |LOCKJAW [INHUMAN]   |
|ARKON               |JARVIS, EDWIN       |
|ARKON               |VISION              |
|ARMSTRONG, MYRA     |GREEN GOBLIN/NORMAN |
+--------------------+--------------------+
only showing top 10 rows
Pairs only in hero-edge.csv (not in our computation): 28,769
+--------------------+--------------------+
|hero1               |hero2               |
+--------------------+--------------------+
|SCARLET WITCH/WANDA |THING/BENJAMIN J. GR|
|BULLSEYE II/BENJAMIN|WATSON-PARKER, MARY |
|HAVOK/ALEX SUMME

In [18]:
# ── (e) Summary ────────────────────────────────────────────────────────────────
print("=" * 55)
print("HERO CO-OCCURRENCE COMPARISON SUMMARY")
print("=" * 55)
print(f"Pairs computed from edges.csv:       {computed_count:,}")
print(f"Pairs in hero-edge.csv:              {hero_edge_normalized.count():,}")
print(f"Pairs in both:                       {in_both.count():,}")
print(f"Only in our computation:             {only_computed.count():,}")
print(f"Only in hero-edge.csv:               {only_in_file.count():,}")
print("=" * 55)
print()
print("Interpretation:")
print("  If 'only in hero-edge.csv' > 0: hero-edge.csv may include")
print("  pairs from a different or extended version of the dataset.")
print("  If 'only in our computation' > 0: we computed more pairs")
print("  possibly because edges.csv is more complete.")

HERO CO-OCCURRENCE COMPARISON SUMMARY
Pairs computed from edges.csv:       171,644
Pairs in hero-edge.csv:              167,219
Pairs in both:                       138,450
Only in our computation:             33,194
Only in hero-edge.csv:               28,769

Interpretation:
  If 'only in hero-edge.csv' > 0: hero-edge.csv may include
  pairs from a different or extended version of the dataset.
  If 'only in our computation' > 0: we computed more pairs
  possibly because edges.csv is more complete.


## Final Summary

| Task | Result |
|------|--------|
| **(a)** Vertices from edges.csv vs nodes.csv | See output above — any discrepancy reported |
| **(b)** Connected Components | N components, largest covers X% of vertices |
| **(c)** Degree / Clustering / Path Length | Sparse graph, small-world structure |
| **(d)** PageRank vs Degree ranking | Heroes with fewer but more central connections rank differently |
| **(e)** Hero pairs vs hero-edge.csv | Differences explained by dataset version |

**AI Usage:**  
Claude (claude.ai) was used throughout this exercise.  
Key prompting steps:
- Understanding GraphFrames API vs GraphX Scala API
- Understanding the Pregel BFS algorithm for path length (why +1, what the map stores, when it stops)
- Understanding PageRank convergence and the role of the reset probability ε
- Understanding connected components — minimum state propagation algorithm
- Understanding clustering coefficient formula and edge case for degree < 2

In [19]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
